# RBI Data Quality Pipeline: Stage 1 (Cleaning)

Public data can quietly go wrong: blank stretches, unit changes, revised
figures. This project builds a pipeline that checks public economic data
for such problems before anyone builds on it.

This notebook is stage one. It takes the RBI's weekly Foreign Exchange
Reserves table (Weekly Statistical Supplement, Table 2, downloaded as an
Excel export), cleans it into 1,334 weekly observations from April 2001
to October 2026, and documents the data quality problems I found along
the way.

Later stages will add automatic quality checks, a benchmark of detection
methods, and a live data source.

## Loading the file

The export from the RBI portal is a report, not a tidy table. Reading it with
`header=None` shows the raw layout: 1,371 rows and 12 columns, with title and
unit lines at the top and the column headers split over two rows.

In [1]:
import pandas as pd
file_name = "weekly_forex_reserves.xlsx"
df = pd.read_excel(file_name,header=None)
print(df.shape)
print(df.head(15))

C:\Users\Ayman\anaconda3\Lib\site-packages\openpyxl\styles\stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")


(1371, 12)
    0                                   1                 2          3   \
0  NaN                                 NaN               NaN        NaN   
1  NaN  Foreign Exchange Reserves - Weekly               NaN        NaN   
2  NaN                                 NaN               NaN        NaN   
3  NaN              Indian Rupees : Crores               NaN        NaN   
4  NaN                                 NaN               NaN        NaN   
5  NaN                US Dollar : Millions               NaN        NaN   
6  NaN                                 NaN               NaN        NaN   
7  NaN                   Year / Week Ended  1 Total Reserves        NaN   
8  NaN                                 NaN     INDIAN RUPEES  US Dollar   
9  NaN                          2026-27                  NaN        NaN   
10 NaN                 2026-10-23 00:00:00        6833504.94  719747.58   
11 NaN                 2026-10-16 00:00:00        6846159.63  721016.25   
12 NaN        

## Removing the junk at the top

The first nine rows hold titles, units and header text, and row 9 is a
fiscal-year label ("2026-27"). The real data starts at row 10, so I dropped
everything above it, plus column 0, which is empty. That left 1,361 rows
and 11 columns.

In [2]:
df = df.iloc[10:]
df = df.drop(columns=0)
df.head()

,1,2,3,4,5,6,7,8,9,10,11
10,2026-10-23 00:00:00,6833504.94,719747.58,5632074.02,593205.38,979339.27,103150.13,176638.98,18604.72,45452.67,4787.35
11,2026-10-16 00:00:00,6846159.63,721016.25,5619655.21,591847.48,1004583.14,105800.05,176395.88,18577.55,45525.4,4791.17
12,2026-10-09 00:00:00,7022417.26,726027.44,5780312.88,597605.42,1016519.74,105094.23,179404.01,18547.92,46180.63,4779.87
13,2026-10-02 00:00:00,7073110.9,734605.64,5822888.87,604746.94,1024611.36,106412.88,179636.24,18656.45,45974.43,4789.37
14,2026-09-25 00:00:00,7164286.55,747557.41,5897822.29,615410.56,1041742.5,108700.99,178655.77,18641.9,46065.99,4803.96


In [3]:
df.shape

(1361, 11)

## Naming the columns

The two header rows are merged cells, so pandas can't use them as names. I
named the 11 columns myself: the week-ending date, then five measures (total
reserves, foreign currency assets, gold, SDRs, IMF reserve position), each in
rupees and then in US dollars. I checked the names against the portal: the
first row should show about 6,833,505 crore rupees and 719,748 million
dollars for 23 October 2026, and it does.

In [4]:
df.columns = ["week_ended","total_reserves_inr","total_reserves_usd",
    "fca_inr","fca_usd",
    "gold_inr","gold_usd",
    "sdr_inr","sdr_usd",
    "imf_inr","imf_usd"]
df.head()

,week_ended,total_reserves_inr,total_reserves_usd,fca_inr,fca_usd,gold_inr,gold_usd,sdr_inr,sdr_usd,imf_inr,imf_usd
10,2026-10-23 00:00:00,6833504.94,719747.58,5632074.02,593205.38,979339.27,103150.13,176638.98,18604.72,45452.67,4787.35
11,2026-10-16 00:00:00,6846159.63,721016.25,5619655.21,591847.48,1004583.14,105800.05,176395.88,18577.55,45525.4,4791.17
12,2026-10-09 00:00:00,7022417.26,726027.44,5780312.88,597605.42,1016519.74,105094.23,179404.01,18547.92,46180.63,4779.87
13,2026-10-02 00:00:00,7073110.9,734605.64,5822888.87,604746.94,1024611.36,106412.88,179636.24,18656.45,45974.43,4789.37
14,2026-09-25 00:00:00,7164286.55,747557.41,5897822.29,615410.56,1041742.5,108700.99,178655.77,18641.9,46065.99,4803.96


In [5]:
df.shape

(1361, 11)

## Cleaning the date column

The date column also holds fiscal-year labels like "2026-27", which are not
weekly observations. I converted the column to dates and dropped whatever
couldn't be converted. That removed 16 rows (1,361 to 1,345), but the oldest
date came out as 2001-02-01, and a week-ending Friday can't fall on the first
of a month. Pandas had read labels like "2001-02" as February 2001, so those
fake rows survived. They had no values in any number column, so I dropped
every row where all the numbers were empty. That removed 11 more rows,
leaving 1,334, and the oldest date is now 2001-04-06.

In [6]:
converted = pd.to_datetime(df["week_ended"],errors = "coerce")
print(converted.isna().sum())

16


In [7]:
df.shape

(1361, 11)

In [8]:
df["week_ended"] = converted
df = df.dropna(subset=["week_ended"])

In [9]:
df.shape

(1345, 11)

In [10]:
print(df["week_ended"].min())
print(df["week_ended"].max())

2001-02-01 00:00:00
2026-10-23 00:00:00


In [11]:
df[df["week_ended"].dt.day == 1]

,week_ended,total_reserves_inr,total_reserves_usd,fca_inr,fca_usd,gold_inr,gold_usd,sdr_inr,sdr_usd,imf_inr,imf_usd
35,2026-05-01,6556403.83,690692.91,5238072.31,551824.65,1093660.46,115215.78,178350.45,18789,46320.61,4863.48
75,2025-08-01,6030417.13,688870.88,5091494.21,581606.57,735336.38,83998.22,162581.59,18571.86,41004.95,4694.23
115,2024-11-01,5735915,682129.96,4959942.75,589849.33,586520.71,69750.57,153197.78,18218.68,36253.76,4311.38
151,2024-03-01,5187143.98,625625.83,4595221.6,554231.45,401430.02,48416.63,150732.63,18179.92,39759.73,4797.83
164,2023-12-01,5031467.8,604042.3,4444765.16,533610.24,394231.41,47328.92,152019.46,18250.49,40451.77,4852.65
177,2023-09-01,4953552.11,598896.64,4389431.78,530690.55,371694.21,44938.53,150491.04,18194.65,41935.08,5072.91
239,2022-07-01,4643704.75,588314.23,4141860.96,524744.52,319054.82,40421.99,143129.45,18133.49,39659.52,5014.23
252,2022-04-01,4592504.47,606475.33,4087017.01,539726.87,323596.28,42733.76,142959.87,18879.12,38931.31,5135.58
279,2021-10-01,4723969.58,637477.28,4264251.26,575450.76,278316.39,37558.15,142576.31,19240.34,38825.62,5228.03
319,2021-01-01,4276975.66,585323.67,3957808.17,541641.68,270554.44,37026.44,11034.9,1510.17,37578.15,5145.38


In [12]:
df = df.dropna(subset = df.columns[1:],how = "all")

In [13]:
df.shape

(1334, 11)

In [14]:
print(df["week_ended"].min())
print(df["week_ended"].max())

2001-04-06 00:00:00
2026-10-23 00:00:00


In [15]:
df.head()

,week_ended,total_reserves_inr,total_reserves_usd,fca_inr,fca_usd,gold_inr,gold_usd,sdr_inr,sdr_usd,imf_inr,imf_usd
10,2026-10-23,6833504.94,719747.58,5632074.02,593205.38,979339.27,103150.13,176638.98,18604.72,45452.67,4787.35
11,2026-10-16,6846159.63,721016.25,5619655.21,591847.48,1004583.14,105800.05,176395.88,18577.55,45525.4,4791.17
12,2026-10-09,7022417.26,726027.44,5780312.88,597605.42,1016519.74,105094.23,179404.01,18547.92,46180.63,4779.87
13,2026-10-02,7073110.9,734605.64,5822888.87,604746.94,1024611.36,106412.88,179636.24,18656.45,45974.43,4789.37
14,2026-09-25,7164286.55,747557.41,5897822.29,615410.56,1041742.5,108700.99,178655.77,18641.9,46065.99,4803.96


In [16]:
print(repr(df["total_reserves_inr"].iloc[0]))

6833504.94


In [17]:
print(df.isna().sum())

week_ended             0
total_reserves_inr     0
total_reserves_usd     0
fca_inr                0
fca_usd                0
gold_inr               0
gold_usd               0
sdr_inr                0
sdr_usd               16
imf_inr               52
imf_usd               52
dtype: int64


## Making the numbers numeric

All the number columns came out as text-like `object` columns, so I converted
them. I counted missing values before and after to make sure the conversion
hadn't wiped out real data. It hadn't: the only missing values left are in
three columns, covered next.

In [18]:
for c in df.columns[1:]:
    df[c] = pd.to_numeric(df[c],errors="coerce")

In [19]:
print(df.isna().sum())

week_ended             0
total_reserves_inr     0
total_reserves_usd     0
fca_inr                0
fca_usd                0
gold_inr               0
gold_usd               0
sdr_inr                0
sdr_usd               16
imf_inr               52
imf_usd               52
dtype: int64


In [20]:
print(df.dtypes)
print(df.head())

week_ended            datetime64[ns]
total_reserves_inr           float64
total_reserves_usd           float64
fca_inr                      float64
fca_usd                      float64
gold_inr                     float64
gold_usd                     float64
sdr_inr                      float64
sdr_usd                      float64
imf_inr                      float64
imf_usd                      float64
dtype: object
   week_ended  total_reserves_inr  total_reserves_usd     fca_inr    fca_usd  \
10 2026-10-23          6833504.94           719747.58  5632074.02  593205.38   
11 2026-10-16          6846159.63           721016.25  5619655.21  591847.48   
12 2026-10-09          7022417.26           726027.44  5780312.88  597605.42   
13 2026-10-02          7073110.90           734605.64  5822888.87  604746.94   
14 2026-09-25          7164286.55           747557.41  5897822.29  615410.56   

      gold_inr   gold_usd    sdr_inr   sdr_usd   imf_inr  imf_usd  
10   979339.27  103150.13  176

In [21]:
missing_imf = df[df["imf_inr"].isna()]
print(missing_imf["week_ended"].min())
print(missing_imf["week_ended"].max())
print(len(missing_imf))

2001-04-06 00:00:00
2002-03-29 00:00:00
52


## What's missing

`imf_inr` and `imf_usd` are blank for 52 weeks in a row, from 2001-04-06 to
2002-03-29, exactly one fiscal year. One unbroken block looks like a
reporting gap and not random corruption. I don't know why RBI left it empty,
so I kept the values missing and did not fill them in.

In [22]:
missing_sdr = df[df["sdr_usd"].isna()]
print(missing_sdr["week_ended"].min())
print(missing_sdr["week_ended"].max())
print(len(missing_sdr))

2006-05-12 00:00:00
2009-08-21 00:00:00
16


In [23]:
print(missing_sdr[["week_ended","sdr_inr","sdr_usd"]])

     week_ended  sdr_inr  sdr_usd
923  2009-08-21      1.0      NaN
924  2009-08-14      1.0      NaN
1000 2008-03-14      1.0      NaN
1001 2008-03-07      1.0      NaN
1002 2008-02-29      1.0      NaN
1003 2008-02-22      1.0      NaN
1004 2008-02-15      1.0      NaN
1088 2006-07-14      2.0      NaN
1089 2006-07-07      2.0      NaN
1090 2006-06-30      2.0      NaN
1091 2006-06-23      2.0      NaN
1092 2006-06-16      2.0      NaN
1093 2006-06-09      2.0      NaN
1095 2006-05-26      2.0      NaN
1096 2006-05-19      2.0      NaN
1097 2006-05-12      2.0      NaN


In [24]:
window = df[(df["week_ended"]>="2006-05-01") & (df["week_ended"] <= "2006-08-01")]
print(window[["week_ended","sdr_inr","sdr_usd"]])

     week_ended  sdr_inr  sdr_usd
1086 2006-07-28     33.0      7.0
1087 2006-07-21     33.0      7.0
1088 2006-07-14      2.0      NaN
1089 2006-07-07      2.0      NaN
1090 2006-06-30      2.0      NaN
1091 2006-06-23      2.0      NaN
1092 2006-06-16      2.0      NaN
1093 2006-06-09      2.0      NaN
1094 2006-06-02      2.0      1.0
1095 2006-05-26      2.0      NaN
1096 2006-05-19      2.0      NaN
1097 2006-05-12      2.0      NaN
1098 2006-05-05     25.0      6.0


In [25]:
print((df["week_ended"].dt.dayofweek != 4).sum())

0


## Final checks

I sorted the table oldest first and reset the row numbers. Every date should
be a week-ending Friday, and the check found 0 dates that aren't. The data
now runs from 2001-04-06 to 2026-10-23, 1,334 weeks in total.

In [26]:
df = df.sort_values(by="week_ended")

In [27]:
df = df.reset_index(drop=True)

In [28]:
df.head()

,week_ended,total_reserves_inr,total_reserves_usd,fca_inr,fca_usd,gold_inr,gold_usd,sdr_inr,sdr_usd,imf_inr,imf_usd
0,2001-04-06,199066.0,42689.0,186344.0,39962.0,12711.0,2725.0,11.0,2.0,NaN,NaN
1,2001-04-13,200258.0,42722.0,187501.0,39987.0,12711.0,2725.0,46.0,10.0,NaN,NaN
2,2001-04-20,200300.0,42765.0,187542.0,40030.0,12711.0,2725.0,47.0,10.0,NaN,NaN
3,2001-04-27,199817.0,42654.0,187059.0,39919.0,12711.0,2725.0,47.0,10.0,NaN,NaN
4,2001-05-04,200356.0,42782.0,187680.0,40077.0,12629.0,2695.0,47.0,10.0,NaN,NaN


In [29]:
df.tail()

,week_ended,total_reserves_inr,total_reserves_usd,fca_inr,fca_usd,gold_inr,gold_usd,sdr_inr,sdr_usd,imf_inr,imf_usd
1329,2026-09-25,7164286.55,747557.41,5897822.29,615410.56,1041742.50,108700.99,178655.77,18641.90,46065.99,4803.96
1330,2026-10-02,7073110.90,734605.64,5822888.87,604746.94,1024611.36,106412.88,179636.24,18656.45,45974.43,4789.37
1331,2026-10-09,7022417.26,726027.44,5780312.88,597605.42,1016519.74,105094.23,179404.01,18547.92,46180.63,4779.87
1332,2026-10-16,6846159.63,721016.25,5619655.21,591847.48,1004583.14,105800.05,176395.88,18577.55,45525.40,4791.17
1333,2026-10-23,6833504.94,719747.58,5632074.02,593205.38,979339.27,103150.13,176638.98,18604.72,45452.67,4787.35


In [30]:
df["implied_rate"] = df["sdr_inr"] * 10 / df["sdr_usd"]
print(df[["week_ended","implied_rate","sdr_inr","sdr_usd"]].dropna(subset="sdr_usd"))

     week_ended  implied_rate    sdr_inr   sdr_usd
0    2001-04-06     55.000000      11.00      2.00
1    2001-04-13     46.000000      46.00     10.00
2    2001-04-20     47.000000      47.00     10.00
3    2001-04-27     47.000000      47.00     10.00
4    2001-05-04     47.000000      47.00     10.00
...         ...           ...        ...       ...
1329 2026-09-25     95.835602  178655.77  18641.90
1330 2026-10-02     96.286400  179636.24  18656.45
1331 2026-10-09     96.724598  179404.01  18547.92
1332 2026-10-16     94.951100  176395.88  18577.55
1333 2026-10-23     94.943100  176638.98  18604.72

[1318 rows x 4 columns]


`sdr_usd` is blank for 16 weeks in three clusters (mid-2006, early 2008 and
August 2009), even though `sdr_inr` has a value in those weeks. The rupee
values there are tiny (1 or 2 crore), while the nearby weeks that do have a
dollar value show 6 or 7. My guess is that amounts this small round away at
the dollar column's precision. On weeks with larger values, the implied rupee-per-dollar rate is stable and
plausible (about 47 in 2001, about 95 in 2026). On very small values it gets
noisy: the first week, with 11 crore and 2 million dollars, implies 55, which
suggests the dollar column loses precision on small amounts. But I haven't confirmed how
RBI encodes these blanks, so this is a hypothesis. I left them missing.

In [31]:
df = df.drop(columns="implied_rate")

The cleaned table is saved as `table2_clean.csv`, ready for the quality
checks in the next stage.

In [32]:
df.to_csv("table2_clean.csv",index=False)

## What I found

- 52 consecutive weeks with no IMF values (April 2001 to March 2002), most
  likely a reporting gap.
- 16 weeks with a blank dollar SDR value despite a rupee value, probably
  rounding of very small amounts (unconfirmed).
- A jump in SDR holdings in the first weeks of the data (11 crore, then 46,
  then 47) that I noticed but haven't investigated.
- My own cleaning step let fake rows through at first. I caught it by
  checking the oldest date.